# 🧪 RS-LiDAR Ablation 1: Phân Tích Prompt Nhạy Cảm & Cắt Lát Bề Mặt Non-Lipschitz
### **Sensitive Prompt Analysis & 1D Landscape Slicing** trên 2x GPU Tesla T4 (Kaggle)

Notebook này được thiết kế chuyên biệt để thực thi **Ablation Study 1** của đề tài RS-LiDAR:
1. **Bản chất của Prompt Nhạy Cảm**: Trong các tác vụ khó của GenEval (như `counting`, `position`, `color_attr`), hàm reward thực nghiệm $R(\mathbf{x})$ (ImageReward, CLIP, HPSv2) xuất hiện các **gai nhọn cục bộ (spurious peaks)** với độ biến thiên tần số cao ($L \gg 1$).
2. **Cơ chế Thất bại của Vanilla LiDAR**: Softmax $\exp(\lambda R)$ bị kéo lệch cực đoan vào các gai ngoại lai này, gây nứt vỡ hình thái ảnh và dị tật chi tiết ở Phase 2.
3. **Sức mạnh của RS-LiDAR**: Tích chập Gaussian $R_\sigma(\mathbf{x}) = \mathbb{E}[R(\mathbf{x} + \sigma \mathbf{u})]$ san phẳng hoàn toàn các gai ngoại lai, áp đặt chặn cứng lý thuyết $\|\nabla R_\sigma\|_2 \le \frac{\Delta R}{\sigma \sqrt{2\pi}}$ (Theorems 3.1 & 3.3).
4. **Cắt lát bề mặt 1D (Landscape Slicing)**: Khảo sát $f(\alpha)$ dọc theo vector vi sai $\mathbf{v} = \frac{\mathbf{x}_{\text{RS}} - \mathbf{x}_{\text{vanilla}}}{\|\mathbf{x}_{\text{RS}} - \mathbf{x}_{\text{vanilla}}\|_2}$ với $\alpha \in [-0.2, 0.2]$.
5. **Tự động trích xuất Top 5 Prompt Divergent**: Xếp hạng và kết xuất bảng ghép ảnh Side-by-Side gồm 5 hàng $\times$ 5 cột, đối chiếu trực quan Unsteered, Vanilla LiDAR, RS-LiDAR và biểu đồ 1D Slicing tương ứng.

## 1. Kiểm Tra Phần Cứng GPU & Cấu Hình Bộ Nhớ
Đảm bảo bạn đã chọn **GPU T4 x2** trong Session options bên phải giao diện Kaggle.

In [ ]:
import os, sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA khả dụng: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"🎮 Số lượng GPU phát hiện: {n_gpus}")
    for i in range(n_gpus):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.2f} GB VRAM)")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng bật GPU trong Settings -> Accelerator -> GPU T4 x2.")

# Chống phân mảnh bộ nhớ VRAM trên GPU T4 16GB
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
!nvidia-smi

## 2. Đồng Bộ Mã Nguồn Repo & Cài Đặt Môi Trường
Cài đặt các gói phụ thuộc và chuẩn bị tiện ích chạy song song 2 GPU hiển thị log trực tiếp.

In [ ]:
import os, shutil, glob, json, random, subprocess, threading, urllib.request

# 1. Đồng bộ repo RS-LiDAR
REPO_DIR = "/kaggle/working/RS-LiDAR"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/leekwanreal/RS-LiDAR.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin main

WORKDIR = f"{REPO_DIR}/Diffusion-LiDAR-Sampling" if os.path.exists(f"{REPO_DIR}/Diffusion-LiDAR-Sampling") else REPO_DIR
os.chdir(WORKDIR)
%cd {WORKDIR}
print("📂 Thư mục làm việc:", os.getcwd())

# 2. Cài đặt các thư viện thiết yếu
!pip install -q --upgrade protobuf
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q image-reward || pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Chuẩn bị file vocab cho HPS v2.1
try:
    import hpsv2
    hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
    os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
    if not os.path.exists(hpsv2_vocab):
        urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)
    print("✅ File vocab HPS v2.1 đã sẵn sàng!")
except Exception as e:
    print(f"Lưu ý khi chuẩn bị HPS vocab: {e}")

# 4. Tiện ích chạy song song 2 GPU hiển thị log trực tiếp
def run_commands_parallel(cmd0, cmd1):
    p0 = subprocess.Popen(cmd0, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    p1 = subprocess.Popen(cmd1, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    def stream_logs(proc, prefix):
        for line in iter(proc.stdout.readline, ''):
            if line.strip():
                print(f"{prefix} {line.strip()}")
        proc.stdout.close()
    t0 = threading.Thread(target=stream_logs, args=(p0, "[GPU 0]"))
    t1 = threading.Thread(target=stream_logs, args=(p1, "[GPU 1]"))
    t0.start(); t1.start()
    t0.join(); t1.join()
    rc0 = p0.wait()
    rc1 = p1.wait()
    if rc0 == 0:
        print("✅ [GPU 0] Tiến trình hoàn thành thành công.")
    else:
        print(f"❌ [GPU 0] Tiến trình dừng với mã lỗi rc={rc0}!")
    if rc1 == 0:
        print("✅ [GPU 1] Tiến trình hoàn thành thành công.")
    else:
        print(f"❌ [GPU 1] Tiến trình dừng với mã lỗi rc={rc1}!")
    if rc0 != 0 or rc1 != 0:
        raise RuntimeError(f"Tiến trình GPU thất bại (GPU 0: {rc0}, GPU 1: {rc1})")

print("\n✅ Môi trường thực nghiệm Ablation 1 đã sẵn sàng 100%!")

## 3. Cấu Hình Tham Số Thực Nghiệm (Ablation 1)
Tất cả tham số dòng lệnh được định nghĩa tập trung tại đây:
- **Cố định Anchor Prompt `#80` (Bench & Ball)**: Luôn luôn giữ lại ca mẫu mực `#80` ở vị trí đầu tiên của Shard 0 (GPU 0).
- **Bốc ngẫu nhiên (Random Exploration)**: Các prompt còn lại được bốc ngẫu nhiên theo `SEED` từ các category phân hóa cao (`two_object`, `colors`, `color_attr`). Mỗi lần đổi `SEED` (vd: 42, 43, 100), hệ thống sẽ khám phá một tập prompt hoàn toàn mới!
- **Phase 1 Lookahead**: $n = 50$ hạt (DPM-5 bước), chấm song song $r_{\text{vanilla}}$ và $r_{\text{RS}}$ ($\|\mathbf{u}\| \sim \mathcal{N}(0, \sigma^2 I), \sigma=1.0, M=4$).
- **Phase 2 Target Guidance**: Text CFG = 12.5 và Particle Guidance scale $s = 12.5$ chuẩn Table 2 SD 1.5, sinh 1 ảnh/seed để đối chiếu 1-1 trực quan và cắt lát 1D.
- **Cắt lát 1D**: $\alpha \in [-0.2, 0.2]$ (41 điểm rời rạc) dọc vector vi sai $\mathbf{v}$.

In [ ]:
# ==============================================================================
# ⚙️ KHU VỰC CẤU HÌNH THAM SỐ ABLATION 1 (TÙY CHỈNH TẠI ĐÂY)
# ==============================================================================
PROMPT_FILE = "prompt_files/geneval_metadata.jsonl"

# 🎯 1. CHỌN DANH SÁCH PROMPTS CỤ THỂ ĐỂ CHẠY (ƯU TIÊN TUYỆT ĐỐI SỐ 1):
#    • Đặt chuỗi ID (ví dụ: "80,98,95,93" hoặc "80,98,95,93,457,138") để CHỈ chạy đúng các prompts này!
#    • Đặt None để tự động chạy theo EXPERIMENT_MODE bên dưới ('curated' hoặc 'original').
PROMPT_INDICES = "80,98,95,93"        # 🌟 Cấu hình mặc định: Top 4 Siêu phẩm đẹp nhất (Bench & Ball, Couch & Horse, Book & Laptop, Couch & Wine)

# 🎯 2. CHẾ ĐỘ THỰC NGHIỆM TỔNG QUÁT (Dùng khi PROMPT_INDICES = None):
#    • 'curated':  CHẾ ĐỘ TUYỂN CHỌN 13 SIÊU PHẨM (LMBDA=1500)
#    • 'original': CHẾ ĐỘ PHÂN TẦNG GỐC (Bốc ngẫu nhiên theo category, LMBDA=5000, Anchor #80, #98)
EXPERIMENT_MODE = "curated"

# ⚡ CỜ ÉP BUỘC CHẠY LẠI & CỜ VẼ BẢNG TOP K:
FORCE_RERUN_INDICES = "80,98,95,93"  # Bắt buộc tính mới hoàn toàn các prompt này với M=16
TOP_K = 10                           # Cờ cấu hình: Vẽ Top K prompts xếp hạng từ cao xuống thấp

if PROMPT_INDICES is not None and str(PROMPT_INDICES).strip() not in ["", "none", "None"]:
    # 🎯 [CHẾ ĐỘ CHỈ ĐỊNH PROMPT CỤ THỂ]
    CURATED_SELECTION = False
    LMBDA = 1500.0                       # Lambda 1500 chống Argmax Collapse
    TARGET_CATEGORIES = None
    PROMPTS_PER_CATEGORY = 0
    ANCHOR_INDICES = None
    RANDOM_SAMPLE = False
    SEED = 42
elif EXPERIMENT_MODE == "curated":
    # 🌟 [CHẾ ĐỘ TUYỂN CHỌN 13 SIÊU PHẨM]
    CURATED_SELECTION = True
    LMBDA = 1500.0
    TARGET_CATEGORIES = None
    PROMPTS_PER_CATEGORY = 0
    ANCHOR_INDICES = None
    RANDOM_SAMPLE = False
    SEED = 42
else:
    # 🔄 [CHẾ ĐỘ PHÂN TẦNG GỐC]
    CURATED_SELECTION = False
    LMBDA = 5000.0
    TARGET_CATEGORIES = "two_object,colors,color_attr"
    PROMPTS_PER_CATEGORY = 4
    ANCHOR_INDICES = "80,98"
    RANDOM_SAMPLE = True
    SEED = 42

NUM_LOOKAHEAD = 50                   # n = 50 hạt lookahead ở Phase 1
LOOKAHEAD_STEPS = 5                  # DPM-5 bước sinh lookahead nhanh
GUIDANCE_SCALE = 12.5                # Hệ số text CFG = 12.5 chuẩn SD 1.5 Table 2
PARTICLE_SCALE = 12.5                # Hệ số closed-form particle guidance s = 12.5
TARGET_STEPS = 50                    # DDIM-50 bước ở Phase 2
SIGMA = 1.0                          # Bán kính làm mịn Gaussian của RS-LiDAR
MC_SAMPLES = 16                      # M = 16 mẫu Monte Carlo tính kỳ vọng (Siêu mượt mà, triệt tiêu phương sai tuyệt đối)
SLICING_MC_SAMPLES = 16              # M = 16 mẫu Monte Carlo cho cắt lát 1D landscape
USE_CRN = True                       # Dùng Common Random Numbers triệt tiêu phương sai Monte Carlo
SLICING_POINTS = 41                  # Số điểm lưới vi phân cắt lát 1D
SLICING_RANGE = 0.2                  # Biên độ vi phân alpha in [-0.2, 0.2]
OVERWRITE = True                     # Đặt True để tính mới hoàn toàn với M=16 cho danh sách prompt đã chọn
# ==============================================================================
TEST_OUTPUT_DIR = "/kaggle/working/results/ablation1_sensitive_landscape"
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

print(f"🎯 Cấu hình Prompts: PROMPT_INDICES = '{PROMPT_INDICES}' (Chế độ chỉ định cụ thể: {PROMPT_INDICES is not None})")
if PROMPT_INDICES:
    print(f"📋 Đang chạy đích danh các prompts: {PROMPT_INDICES}")
elif CURATED_SELECTION:
    print("🌟 Đang kích hoạt 13 Siêu phẩm Chủ-Khách (#80, #98, #95, #93, #97, #109, #110, #125, #138, #142, #156, #457, #464)")
else:
    print(f"⚙️ Cấu hình Category: '{TARGET_CATEGORIES}' | {PROMPTS_PER_CATEGORY} prompts/category")
    print(f"⚓ Anchor cố định: #{ANCHOR_INDICES} | Random Sample: {RANDOM_SAMPLE} (Seed={SEED})")
print(f"⚡ Force Rerun Indices: '{FORCE_RERUN_INDICES}' | Overwrite: {OVERWRITE}")
print(f"🏆 Top K Visualization: Top {TOP_K} xếp từ cao xuống thấp")
print(f"🌡️ Softmax Lambda: {LMBDA} (Lưu ý: 1500.0 giúp chống trùng ảnh dist=0; 5000.0 là mức chuẩn ban đầu)")
print(f"🔬 Phase 1: {NUM_LOOKAHEAD} hạt lookahead (DPM-{LOOKAHEAD_STEPS}) | Sigma = {SIGMA} | M = {MC_SAMPLES} | CRN = {USE_CRN}")
print(f"🎯 Phase 2: Text CFG = {GUIDANCE_SCALE} | Particle scale s = {PARTICLE_SCALE} | Target steps = {TARGET_STEPS}")
print(f"📐 1D Slicing: {SLICING_POINTS} điểm trên dải alpha in [{-SLICING_RANGE}, {SLICING_RANGE}] | M = {SLICING_MC_SAMPLES}")
print(f"📁 Thư mục lưu kết quả: {TEST_OUTPUT_DIR}")


## 4. Tiền Tải Trọng Số & Thực Thi Song Song 2x GPU T4
Quá trình tiền tải model weights vào cache ổ đĩa để tránh xung đột đa tiến trình, sau đó khởi chạy Shard 0 (GPU 0) và Shard 1 (GPU 1) song song.

In [ ]:
# 1. Tự động kiểm tra & tự phục hồi thư mục làm việc chuẩn
if not os.path.exists("test_sensitive_prompt_landscape.py"):
    for cand in ["/kaggle/working/RS-LiDAR/Diffusion-LiDAR-Sampling", "/kaggle/working/RS-LiDAR", "."]:
        if os.path.exists(os.path.join(cand, "test_sensitive_prompt_landscape.py")):
            os.chdir(cand)
            print("📂 Đã chuyển đến thư mục làm việc:", os.getcwd())
            break

# 2. Dọn dẹp tiến trình cũ nếu có và giải phóng VRAM
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
!pkill -f "test_sensitive_prompt_landscape.py" || true

# 3. Tiền tải trọng số Stable Diffusion 1.5, ImageReward, CLIP
print("📥 Đang kiểm tra & tiền tải toàn bộ model weights chuẩn vào cache ổ đĩa...")
from huggingface_hub import snapshot_download
snapshot_download(
    "runwayml/stable-diffusion-v1-5",
    allow_patterns=["model_index.json", "*scheduler/*", "*tokenizer*/*", "*text_encoder*/*model.safetensors", "*unet*/*diffusion_pytorch_model.safetensors", "*vae*/*diffusion_pytorch_model.safetensors"],
    ignore_patterns=["*.ckpt", "*pruned*", "*non_ema*", "safety_checker/*", "*.bin", "*fp16*"]
)

# ImageReward weights
ir_path = os.path.expanduser("~/.cache/ImageReward/ImageReward.pt")
if not os.path.exists(ir_path):
    os.makedirs(os.path.dirname(ir_path), exist_ok=True)
    try:
        urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/ImageReward.pt", ir_path)
        urllib.request.urlretrieve("https://huggingface.co/THUDM/ImageReward/resolve/main/med_config.json", os.path.join(os.path.dirname(ir_path), "med_config.json"))
        print("✅ Đã tiền tải xong ImageReward weights!")
    except Exception as e:
        print(f"Lưu ý tải ImageReward: {e}")

# OpenAI CLIP ViT-L/14 Weights
try:
    import clip
    _ = clip.load("ViT-L/14", device="cpu")
    print("✅ Trọng số OpenAI CLIP ViT-L/14 đã sẵn sàng trong cache!")
except Exception as e:
    print(f"Lưu ý nạp CLIP: {e}")

# 4. Khởi chạy thực nghiệm Ablation 1 song song 2 GPU
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
ov_flag = "--overwrite" if OVERWRITE else "--resume"
crn_flag = "--use_crn" if USE_CRN else "--no_crn"
rnd_flag = "--random_sample" if RANDOM_SAMPLE else "--no_random_sample"
anchor_flag = f'--anchor_indices "{ANCHOR_INDICES}"' if ANCHOR_INDICES else ""
force_flag = f'--force_rerun_indices "{FORCE_RERUN_INDICES}"' if FORCE_RERUN_INDICES else ""
lmbda_flag = f"--lmbda {LMBDA}"

# Ưu tiên chỉ định prompt cụ thể nếu có
if PROMPT_INDICES is not None and str(PROMPT_INDICES).strip() not in ["", "none", "None"]:
    prompt_flags = f'--prompt_indices "{PROMPT_INDICES}"'
elif CURATED_SELECTION:
    prompt_flags = "--curated_selection"
else:
    prompt_flags = f'--categories "{TARGET_CATEGORIES}" --prompts_per_category {PROMPTS_PER_CATEGORY} {anchor_flag} {rnd_flag} --seed {SEED}'

print(f"\n🚀 Bắt đầu thực nghiệm Ablation 1 trên {n_gpus} GPU...\n")

if n_gpus >= 2:
    print("⚡ [2 GPU] Đang chạy song song trên GPU 0 (Shard 0) và GPU 1 (Shard 1)...")
    cmd0 = f"""python -u test_sensitive_prompt_landscape.py \
        --prompts_file "{PROMPT_FILE}" \
        {prompt_flags} \
        {force_flag} \
        {lmbda_flag} \
        --top_k {TOP_K} \
        --num_lookahead {NUM_LOOKAHEAD} \
        --lookahead_steps {LOOKAHEAD_STEPS} \
        --guidance_scale {GUIDANCE_SCALE} \
        --particle_scale {PARTICLE_SCALE} \
        --target_steps {TARGET_STEPS} \
        --sigma {SIGMA} \
        --mc_samples {MC_SAMPLES} \
        --slicing_points {SLICING_POINTS} \
        --slicing_range {SLICING_RANGE} \
        --slicing_mc_samples {SLICING_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 0 --gpu_id 0 {crn_flag} {ov_flag}"""

    cmd1 = f"""python -u test_sensitive_prompt_landscape.py \
        --prompts_file "{PROMPT_FILE}" \
        {prompt_flags} \
        {force_flag} \
        {lmbda_flag} \
        --top_k {TOP_K} \
        --num_lookahead {NUM_LOOKAHEAD} \
        --lookahead_steps {LOOKAHEAD_STEPS} \
        --guidance_scale {GUIDANCE_SCALE} \
        --particle_scale {PARTICLE_SCALE} \
        --target_steps {TARGET_STEPS} \
        --sigma {SIGMA} \
        --mc_samples {MC_SAMPLES} \
        --slicing_points {SLICING_POINTS} \
        --slicing_range {SLICING_RANGE} \
        --slicing_mc_samples {SLICING_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" \
        --num_shards 2 --shard_id 1 --gpu_id 1 {crn_flag} {ov_flag}"""
    run_commands_parallel(cmd0, cmd1)
else:
    print("⚡ [1 GPU] Đang chạy trên 1 GPU...")
    !python -u test_sensitive_prompt_landscape.py \
        --prompts_file "{PROMPT_FILE}" \
        {prompt_flags} \
        {force_flag} \
        {lmbda_flag} \
        --top_k {TOP_K} \
        --num_lookahead {NUM_LOOKAHEAD} \
        --lookahead_steps {LOOKAHEAD_STEPS} \
        --guidance_scale {GUIDANCE_SCALE} \
        --particle_scale {PARTICLE_SCALE} \
        --target_steps {TARGET_STEPS} \
        --sigma {SIGMA} \
        --mc_samples {MC_SAMPLES} \
        --slicing_points {SLICING_POINTS} \
        --slicing_range {SLICING_RANGE} \
        --slicing_mc_samples {SLICING_MC_SAMPLES} \
        --output_dir "{TEST_OUTPUT_DIR}" {crn_flag} {ov_flag}

print("\n✅ HOÀN THÀNH TÍNH TOÁN! Tiến hành gộp shard...")


## 5. Tự Động Gộp Kết Quả Shards & Sắp Xếp Xếp Hạng Top 10
Lệnh `--merge_shards` tự động tìm tất cả checkpoint shards, gộp lại thành 1 danh sách duy nhất, tính toán các chỉ số Divergence Score (Rescue Score & Smoothness Score), sắp xếp giảm dần và trích xuất bảng Top 10 kèm các đồ họa Side-by-Side.

In [ ]:
print("🔄 Đang gộp kết quả từ các shard checkpoints...")
!python -u test_sensitive_prompt_landscape.py --merge_shards --output_dir "{TEST_OUTPUT_DIR}" --top_k {TOP_K}
print("✅ Hoàn tất gộp dữ liệu và xuất báo cáo!")


## 6. Hiển Thị Bảng Top 10 & Đồ Họa Side-by-Side Dual-Panel Visual Comparison
Hiển thị trực tiếp bảng xếp hạng Top 10 và các Panel đồ họa phân giải cao ngay trong notebook:

In [ ]:
import pandas as pd, glob, os
from IPython.display import display, Image, Markdown

# 1. Bảng Top K Failure Rescue Prompts
rescue_csv = os.path.join(TEST_OUTPUT_DIR, f"ablation1_top{TOP_K}_rescue.csv")
if not os.path.exists(rescue_csv):
    rescue_csv = os.path.join(TEST_OUTPUT_DIR, "ablation1_top5_rescue.csv")
if not os.path.exists(rescue_csv):
    rescue_csv = os.path.join(TEST_OUTPUT_DIR, "ablation1_top5_divergence.csv") # fallback
if os.path.exists(rescue_csv):
    df_rescue = pd.read_csv(rescue_csv)
    print("=" * 115)
    print(f"🏆 BẢNG 1: TOP {len(df_rescue)} PROMPTS CỨU CÁNH THẤT BẠI (FAILURE-RESCUE RANKING)")
    print("=" * 115)
    display(df_rescue)

# 2. Bảng Top K Lipschitz Smoothness Prompts
smoothness_csv = os.path.join(TEST_OUTPUT_DIR, f"ablation1_top{TOP_K}_smoothness.csv")
if not os.path.exists(smoothness_csv):
    smoothness_csv = os.path.join(TEST_OUTPUT_DIR, "ablation1_top5_smoothness.csv")
if os.path.exists(smoothness_csv):
    df_smoothness = pd.read_csv(smoothness_csv)
    print("\n" + "=" * 115)
    print(f"🛡️ BẢNG 2: TOP {len(df_smoothness)} PROMPTS LÀM PHẲNG LIPSCHITZ MẠNH NHẤT (SMOOTHNESS GAIN)")
    print("=" * 115)
    display(df_smoothness)

# 3. Bảng Tổng Hợp Toàn Bộ Prompts
summary_csv = os.path.join(TEST_OUTPUT_DIR, "ablation1_summary.csv")
if os.path.exists(summary_csv):
    df_summary = pd.read_csv(summary_csv)
    print("\n" + "=" * 115)
    print(f"📊 BẢNG TỔNG HỢP TOÀN BỘ {len(df_summary)} PROMPTS:")
    print("=" * 115)
    display(df_summary.head(TOP_K))

# 4. Hiển thị Bảng Ghép Đồ Họa Side-by-Side: Failure Rescue Panel
rescue_panel = os.path.join(TEST_OUTPUT_DIR, f"ablation1_top{TOP_K}_rescue_panel.png")
if not os.path.exists(rescue_panel):
    rescue_panel = os.path.join(TEST_OUTPUT_DIR, "ablation1_top5_rescue_panel.png")
if not os.path.exists(rescue_panel):
    rescue_panel = os.path.join(TEST_OUTPUT_DIR, "ablation1_top5_divergence_panel.png") # fallback
if os.path.exists(rescue_panel):
    print(f"\n🖼️ [PANEL 1] ĐỒ HỌA TOP {TOP_K} FAILURE RESCUE (SIDE-BY-SIDE PANEL):")
    display(Image(filename=rescue_panel))

# 5. Hiển thị Bảng Ghép Đồ Họa Side-by-Side: Lipschitz Smoothness Panel
smoothness_panel = os.path.join(TEST_OUTPUT_DIR, f"ablation1_top{TOP_K}_smoothness_panel.png")
if not os.path.exists(smoothness_panel):
    smoothness_panel = os.path.join(TEST_OUTPUT_DIR, "ablation1_top5_smoothness_panel.png")
if os.path.exists(smoothness_panel):
    print(f"\n🖼️ [PANEL 2] ĐỒ HỌA TOP {TOP_K} LIPSCHITZ SMOOTHNESS (SIDE-BY-SIDE PANEL):")
    display(Image(filename=smoothness_panel))

# 6. Hiển thị Đồ thị Cắt lát 1D Landscape Curves
curves_img = os.path.join(TEST_OUTPUT_DIR, "ablation1_landscape_1d_curves.png")
if os.path.exists(curves_img):
    print("\n📈 ĐỒ THỊ TỔNG HỢP 1D LANDSCAPE CURVES:")
    display(Image(filename=curves_img))


## 7. Đóng Gói Toàn Bộ Kết Quả (1-Click Download Cho Save Version)
Nén toàn bộ các file CSV, JSON metrics, ảnh PNG sinh ra và đồ thị so sánh vào `/kaggle/working/results_ablation1_sensitive_landscape.zip` để bạn có thể tải về trực tiếp từ tab **Output** bên phải Kaggle.

In [ ]:
zip_path = "/kaggle/working/results_ablation1_sensitive_landscape.zip"
print(f"📦 Đang nén toàn bộ kết quả Ablation 1 vào {zip_path}...")

!zip -r -q {zip_path} {TEST_OUTPUT_DIR}

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"🎉 ĐÓNG GÓI THÀNH CÔNG! Dung lượng file: {size_mb:.2f} MB")
    print(f"📁 Đường dẫn file zip: {zip_path}")
    print("👉 Bạn có thể tải file này trực tiếp từ cột 'Output' bên phải giao diện Kaggle!")
else:
    print("⚠️ Không tìm thấy file zip được tạo.")